# Visualizing Block Sizes in SCRIPT Encoding

This notebook explores and visualizes the block sizes for SCRIPT encoding

In [1]:
import matplotlib.pyplot as plt
from script_bpe.pretokenize.scriptencoding import ScriptEncodingV1, ScriptEncodingV2
import pandas as pd
import numpy as np
import math
import copy


ScriptEncodingNSV1 = copy.deepcopy(ScriptEncodingV1)    
ScriptEncodingNSV1.largest_block = ("Han", "LM")
ScriptEncodingNSV1.blocks = []
ScriptEncodingNSV1.model_post_init(None)


ScriptEncodingNSV2 = copy.deepcopy(ScriptEncodingV2)
ScriptEncodingNSV2.largest_block = ("Han", "LM")
ScriptEncodingNSV2.blocks = []
ScriptEncodingNSV2.model_post_init(None)


script_encoding = ScriptEncodingNSV2
blocks = sorted(script_encoding.blocks, key=lambda x: len(x.chars), reverse=True)
block_sizes = [len(block.chars) for block in blocks]
block_labels = [f"{block.script}-{block.category}" for block in blocks]



In [2]:
print("Number of blocks in ScriptEncoding V1:", len(ScriptEncodingV1.blocks))
print("Number of script/category combinations V1:", len(set([(block.script, block.category) for block in ScriptEncodingV1.blocks])))
print("Number of blocks in ScriptEncoding V2:", len(ScriptEncodingV2.blocks))
print("Number of script/category combinations V2:", len(set([(block.script, block.category) for block in ScriptEncodingV2.blocks])))

Number of blocks in ScriptEncoding V1: 468
Number of script/category combinations V1: 381
Number of blocks in ScriptEncoding V2: 262
Number of script/category combinations V2: 174


In [4]:
# Initialize ScriptEncoding and extract block data
script_encoding = ScriptEncodingNSV2
blocks = sorted(script_encoding.blocks, key=lambda x: len(x.chars), reverse=True)
block_sizes = [len(block.chars) for block in blocks]
block_labels = [f"{block.script}-{block.category}" for block in blocks]


In [7]:
# make df with script, supercategory, and block size
df_blocks = pd.DataFrame(
    {
        "Script": [block.script.replace("_", " ") for block in blocks],
        "Supercategory": [block.category for block in blocks],
        "Size": [len(block.chars) for block in blocks],
        "Chars": [block.chars[:10] for block in blocks],
    }
)
# df_blocks = df_blocks[df_blocks.Supercategory=='N']
# make script/supercat index
df_blocks.set_index(["Script", "Supercategory"]).style.set_caption("Table 1: Script Blocks and Sizes")

,,Size,Chars
Script,Supercategory,,
Han,LM,98687,々〻㐀㐁㐂㐃㐄㐅㐆㐇
Hangul,LM,11677,ᄀᄁᄂᄃᄄᄅᄆᄇᄈᄉ
✭,PSF,8182,¡¢£¤¥¦§¨©«
Tangut,✭,6914,𖿠𗀀𗀁𗀂𗀃𗀄𗀅𗀆𗀇𗀈
Egyptian Hieroglyphs,✭,5105,𓀀𓀁𓀂𓀃𓀄𓀅𓀆𓀇𓀈𓀉
Latin,LM,1448,ABCDEFGHIJ
Arabic,LM,1254,ـؘؐؑؒؓؔؕؖؗ
Cuneiform,✭,1234,𒀀𒀁𒀂𒀃𒀄𒀅𒀆𒀇𒀈𒀉
Yi,✭,1220,ꀀꀁꀂꀃꀄꀅꀆꀇꀈꀉ


In [6]:
bins = [0.5, 2.5, 5.5, 10.5, 50.5, 100.5, 500.5, 5000.5, 1000000]

bin_labels = [
    f"{math.ceil(bins[i])}-{math.floor(bins[i + 1])}" if i < len(bins) - 2 else f"{math.ceil(bins[i])}+"
    for i in range(len(bins) - 1)
]

category_colors = {"LM": "#3878a6", "PS": "#5b9ab5", "N": "#45a36a", "Z": "#f0f1f2", "C": "#8c8c8c"}

# Compute counts per bin for each supercategory
stack_counts = {}
for cat in category_colors:
    cat_data = df_blocks[df_blocks["Supercategory"] == cat]["Size"]
    counts_cat, _ = np.histogram(cat_data, bins=bins)
    stack_counts[cat] = counts_cat

x_positions = np.arange(len(bin_labels))
plt.figure(figsize=(12, 8))
bottom = np.zeros(len(bin_labels))  # to keep track of the lower bound of each stack

# Loop through each category and plot its counts on top of what was already plotted.
for cat, color in category_colors.items():
    counts = stack_counts[cat]
    bars = plt.bar(x_positions, counts, bottom=bottom, color=color, edgecolor="black", linewidth=1.2, label=cat)
    # Annotate each segment in the stacked bar with its count
    for i, bar in enumerate(bars):
        count = counts[i]
        if count > 1:
            # Place the text at the center of the bar segment
            plt.text(
                bar.get_x() + bar.get_width() / 2,
                bottom[i] + count / 2,
                str(count),
                ha="center",
                va="center",
                fontsize=10,
                color="black",
            )
    # Update bottom to include the current category counts
    bottom += counts

plt.xticks(x_positions, bin_labels, rotation=45, fontsize=12)
plt.yticks(fontsize=12)
plt.xlabel("Block Size", fontsize=14)
plt.ylabel("Count", fontsize=14)
plt.legend(title="Supercategory", fontsize=12)
plt.tight_layout()

NameError: name 'df_blocks' is not defined